# Lecture 17 - Heaps and sorting

**Lecture Topic**: A tree that lives inside an array and hands you the smallest item instantly, and how the sorts you call every day actually work.

## Announcements

- Module C continues.
- Keep going on your project work.
- ClassBuzz poll for today: `XXXX`

## Recap: last time

- A **binary search tree**: smaller left, larger right, so each step halves the search.
- Cost is the **height**: `O(log n)` balanced, `O(n)` degenerate; real systems keep trees balanced.
- **Traversal**, **binary search** on sorted arrays, and **kd-trees** for nearest neighbours.

Today: the **heap**, a different tree with a different job, and **sorting**, which you almost never write but choose.

Today:
 - Heaps and priority queues
 - How a heap lives in a plain array
 - Top-k without a full sort
 - The classic sorts, compared
 - Stability, reproducibility, and what the libraries actually use
 - Sorting data that does not fit in memory

# Heaps: fast access to the smallest (or largest)

Sometimes you do not need the data sorted, you just need the **minimum** (or maximum) quickly, over and over. A hospital triage queue is the picture: the next patient is the most severe, regardless of arrival order.

A sorted list is not the answer: it gives the min in O(1), but every insert is O(n) to keep it sorted. A **heap** gives you both insert and remove-min in **O(log n)**.

A heap is a **complete binary tree** (every level full, last level filled left to right) obeying the **heap property**:

 - **min-heap**: every parent is less than or equal to its children, so the smallest value is always at the root.

Two operations, each O(log n):

 - **insert**: put the item at the end, then *bubble it up* past any larger parent.
 - **remove-min**: take the root, move the last item to the root, then *bubble it down* past its smaller child.

## A heap lives in a plain array

Because the tree is *complete*, it needs no nodes or pointers at all. Lay the levels out left to right in an array, and the parent/child links become arithmetic:

 - node at index `i` has children at `2*i + 1` and `2*i + 2`,
 - and its parent is at `(i - 1) // 2`.

That is why Python's `heapq` operates directly on an ordinary list.

In [ ]:
import heapq

h = []
for x in [5, 3, 8, 1, 9, 2]:
    heapq.heappush(h, x)          # min-heap, stored as a list

print("heap array:", h)            # not sorted, but h[0] is always the minimum
print("the min:   ", h[0])
print("pop order: ", [heapq.heappop(h) for _ in range(len(h))])  # comes out ascending

### Question 1

A heap is stored in an array. The node at index `i` has its parent at which index?

 - A. `i - 2`
 - B. `i / 2`
 - C. `(i - 1) // 2`
 - D. `2 * i`

**C, `(i - 1) // 2`.** Children sit at `2*i + 1` and `2*i + 2`, so inverting that puts the parent at `(i - 1) // 2`. Completeness is what makes this arithmetic exact, and it is why a heap needs only an array, not a tree of objects.

## How insert and remove actually work

Both operations keep the heap property by moving one value along a single root-to-leaf path, which is why each is `O(log n)`.

 - **Insert:** put the new value at the end, then *bubble it up*, swapping with its parent while it is smaller.
 - **Remove-min:** take the root, move the last value into its place, then *bubble it down*, swapping with its smaller child until it settles.

In [ ]:
class MinHeap:
    def __init__(self):
        self.a = []

    def push(self, x):
        self.a.append(x)                        # put the new value at the end
        i = len(self.a) - 1
        while i > 0 and self.a[(i - 1) // 2] > self.a[i]:    # bubble up past a larger parent
            self.a[i], self.a[(i - 1) // 2] = self.a[(i - 1) // 2], self.a[i]
            i = (i - 1) // 2

    def pop_min(self):
        self.a[0], self.a[-1] = self.a[-1], self.a[0]        # last value moves to the root
        smallest = self.a.pop()
        i, n = 0, len(self.a)
        while True:                             # bubble down past the smaller child
            lo = i
            for child in (2 * i + 1, 2 * i + 2):
                if child < n and self.a[child] < self.a[lo]:
                    lo = child
            if lo == i:
                break
            self.a[i], self.a[lo] = self.a[lo], self.a[i]
            i = lo
        return smallest

h = MinHeap()
for x in [5, 3, 8, 1, 9, 2]:
    h.push(x)
    print(f"after push {x}: {h.a}")
print("pop_min:", h.pop_min(), "-> array now", h.a)

### Question 2

Reading each array as a complete binary tree (index `i` has children at `2*i + 1` and `2*i + 2`), which one is a valid **min-heap**, where every parent is less than or equal to its children?

 - A. `[1, 3, 2, 7, 4, 5]`
 - B. `[1, 2, 5, 3, 4, 0]`
 - C. `[3, 1, 2]`
 - D. More than one

**A.** In B, the `5` at index 2 has a child `0` at index 5, and `5 > 0` breaks the rule. In C, the root `3` is larger than both of its children. Only A has every parent no larger than its children. The heap property is local, parent against children, which is what keeps insert and remove cheap.

## Top-k: the heap's best trick

A common question on big data is "what are the top 10?" You do **not** need to sort all `n` items to answer it. Keep a heap of only the `k` best seen so far, which costs `O(n log k)`, far less than the `O(n log n)` of a full sort when `k` is small.

In [ ]:
import heapq
import random
import time

random.seed(0)
data = [random.random() for _ in range(2_000_000)]

start = time.perf_counter()
top_sorted = sorted(data, reverse=True)[:10]        # sort everything, keep 10
t_sort = time.perf_counter() - start

start = time.perf_counter()
top_heap = heapq.nlargest(10, data)                 # heap of size 10
t_heap = time.perf_counter() - start

print(f"full sort then slice: {t_sort:.3f} s")
print(f"heapq.nlargest(10):   {t_heap:.3f} s")
print(f"same top 10? {top_sorted == top_heap}")

Same ten values, less work, because `nlargest` never orders the other ~2 million items. When you want "the biggest few" rather than "everything in order", reach for `heapq.nlargest` / `nsmallest`.

# Sorting

When you *do* need everything in order, there are two things to weigh: **time** and the **extra memory** the sort needs. Here are the classics.

 - **Selection sort**: repeatedly find the smallest of the rest and swap it into place. Always `O(n^2)`, in place.
 - **Insertion sort**: grow a sorted prefix, sliding each new item back to its spot. `O(n)` on nearly-sorted data, `O(n^2)` worst, in place.
 - **Merge sort**: split in half, sort each, merge. Always `O(n log n)`, but needs `O(n)` extra space.
 - **Quick sort**: partition around a pivot, recurse on each side. `O(n log n)` average, `O(n^2)` worst, `O(log n)` space.
 - **Heap sort**: build a heap, pop the min `n` times. Always `O(n log n)`, in place.

| Sort | Best | Worst | Extra space | Stable? |
|---|---|---|---|---|
| Selection | O(n^2) | O(n^2) | O(1) | no |
| Insertion | O(n) | O(n^2) | O(1) | yes |
| Merge | O(n log n) | O(n log n) | O(n) | yes |
| Quick | O(n log n) | O(n^2) | O(log n) | no |
| Heap | O(n log n) | O(n log n) | O(1) | no |

No single winner: merge sort guarantees `O(n log n)` but spends memory; quick sort is usually fastest in practice but has a bad worst case; heap sort is `O(n log n)` in place but not stable.

### Question 3

What are the **best-case** and **worst-case** time complexities of **merge sort**?

 - A. O(n), O(1)
 - B. O(n log n), O(n log n)
 - C. O(n), O(log n)
 - D. O(n^2), O(n)

**B.** Merge sort always splits in half (about `log n` levels) and does `O(n)` work merging at each level, so it is `O(n log n)` no matter the input. That predictability, plus stability, is why it underlies the sorts the libraries ship, even though it costs `O(n)` extra memory.

## Why no comparison sort beats n log n

There is a floor worth knowing. A sort that works by comparing pairs of elements has to tell apart all `n!` possible orderings of the input, and each comparison has two outcomes, so at best it halves the orderings still in play. To single out one ordering from `n!` you need at least about `log2(n!)` comparisons, which works out to roughly `n log n`. So `O(n log n)` is the best any comparison sort can do.

Counting sort and radix sort reach `O(n)` only because they skip comparison: they read the digits or buckets of the keys directly, which is why they apply to specific kinds of keys rather than anything you can put in order.

# Stability and reproducibility

A sort is **stable** if it keeps equal elements in their original order. That sounds like a detail until you sort by more than one key, or need a pipeline to give the same answer every run.

Multi-key sorting is built on stability: to sort by city, then name within city, you sort by name first, then do a **stable** sort by city.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "city": ["SD", "LA", "SD", "LA"],
    "name": ["Ana", "Bo", "Cy", "Al"],
})

step1 = df.sort_values("name")                      # Al, Ana, Bo, Cy
final = step1.sort_values("city", kind="stable")    # stable keeps names in order within a city
print(final.to_string(index=False))

Within each city the names stayed alphabetical, because the stable sort did not disturb the order the first sort established. An **unstable** sort (NumPy's and pandas' default `quicksort`) could scramble the ties, and worse, it might scramble them *differently* on another machine or another version, so a run is not reproducible. When ties must break the same way every time, ask for a stable sort.

# What Python, NumPy and pandas actually use

You rarely implement a sort. You pick one, and the pick is about stability and memory, not the algorithm.

 - **Python** `sorted` / `list.sort`: **Timsort**, a hybrid of merge and insertion sort. Stable, `O(n log n)`, and specially fast on the partially-ordered data you meet in practice.
 - **NumPy** `np.sort`: defaults to an introsort (quicksort that falls back to heap sort to dodge the `O(n^2)` case). Not stable unless you pass `kind="stable"`.
 - **pandas** `sort_values`: uses NumPy underneath; default `quicksort`, with `kind="stable"` when you need stable ties.

In [ ]:
import numpy as np

# Python's sorted is stable: ties keep input order
records = [("b", 1), ("a", 2), ("b", 3), ("a", 4)]
print("sorted by letter:", sorted(records, key=lambda r: r[0]))

# NumPy's default is not guaranteed stable; ask for it explicitly
keys = np.array([2, 1, 2, 1, 2, 1])
print("default argsort:", np.argsort(keys))
print("stable argsort: ", np.argsort(keys, kind="stable"))   # equal keys keep input order

Both came out the same here, but only the stable sort *promises* it: equal keys keep their input order, so the three `1`s stay as positions `1, 3, 5` and the `2`s as `0, 2, 4`. The default makes no such promise, and on larger data quicksort does reorder ties, potentially differently on another machine or version. When ties must break the same way every run, that is exactly what `kind="stable"` buys you.

## The same thing on real data

Here is where tie order bites in practice: the ten busiest pickup-to-dropoff routes in the taxi sample. Several routes are tied on trip count, so the ranked list depends on how those ties are broken.

In [ ]:
import pandas as pd

taxi = pd.read_csv("../data/2019_Yellow_Taxi_Trip_Data.csv")
counts = taxi.groupby(["pulocationid", "dolocationid"]).size().reset_index(name="trips")

stable = counts.sort_values("trips", ascending=False, kind="stable").head(10)
quick  = counts.sort_values("trips", ascending=False, kind="quicksort").head(10)

same_members = (set(map(tuple, stable[["pulocationid", "dolocationid"]].to_numpy())) ==
                set(map(tuple, quick[["pulocationid", "dolocationid"]].to_numpy())))
print("trip counts at ranks 8-10:", sorted(counts["trips"], reverse=True)[7:10])
print("same ten routes? ", same_members)
print("same order?      ", stable.reset_index(drop=True).equals(quick.reset_index(drop=True)))

The two sorts return the same ten routes. But the routes tied at the same trip count (here three of them, all at the bottom of the list) have no guaranteed order under the default sort, so a "top 10" table built this way can change between runs or library versions with no change in the data. For a reproducible ranking, sort stably or add an explicit tie-breaker, for example `sort_values(["trips", "pulocationid", "dolocationid"], ...)`.

## One anti-pattern to watch

A related trap in agent code is sorting inside a loop: re-sorting a growing list every time an item is added, or calling `sort` again on every pass. That turns an `O(n log n)` job into `O(n^2 log n)`. If you only need the largest few, use a heap (`heapq.nlargest`) as above; if you need everything in order, sort once at the end.

# Sorting data that does not fit in memory

Everything so far assumed the data fits in RAM. When it does not, you cannot load it to sort it. The answer is an **external (out-of-core) merge sort**:

 1. Read the data in chunks that *do* fit, sort each chunk, and write the sorted **runs** to disk.
 2. **Merge** the sorted runs together in a single streaming pass.

The merge step needs almost no memory: it only ever looks at the current front item of each run, which is exactly a heap of size "number of runs". `heapq.merge` does this lazily.

In [ ]:
import heapq

# pretend each 'run' is a sorted chunk already written to disk
run_a = [1, 4, 7, 10]
run_b = [2, 3, 8]
run_c = [0, 5, 6, 9]

merged = heapq.merge(run_a, run_b, run_c)    # a lazy iterator, not a built list
print("streamed in sorted order:", list(merged))

`heapq.merge` yields one item at a time and never holds all the data at once, so you can merge runs far larger than memory. This is how databases and tools like the Unix `sort` command sort tables bigger than RAM, and it is the memory-budget lesson from the complexity lecture made concrete: the binding limit was space, and the algorithm is chosen to respect it.

# Recap

 - A **heap** is a complete binary tree in an array; `O(log n)` insert and remove-min, instant access to the extreme.
 - **Top-k** uses a size-`k` heap: `O(n log k)`, no full sort needed (`heapq.nlargest`).
 - The classic sorts trade **time** against **extra memory**; merge is predictable, quick is usually fastest, heap is in-place.
 - **Insert** and **remove-min** keep the heap property by bubbling one value up or down a single path, so each is `O(log n)`.
 - **Stability** keeps ties in order; it is what multi-key sorts and reproducible rankings depend on, so sort stably or add a tie-breaker.
 - No comparison sort beats `O(n log n)`; counting and radix go faster only by skipping comparison.
 - **Python** uses Timsort (stable); **NumPy/pandas** default to quicksort (`kind="stable"` when you need it).
 - **Out-of-core merge** sorts data larger than memory with `heapq.merge`.

**Next time (S20): graphs and DAGs** - the most general structure, and the dependency graphs behind every data pipeline.